In [1]:
import torch as t
import pandas as pd
from itertools import product
import numpy as np
from utils import collect_acts
from utils import DataManager
from generate_acts import load_model
from probes import LRProbe, MMProbe
import argparse
import configparser
from nie_utils import compute_nie_dataframe
from interventions import intervention_experiment, prepare_data
from prompts import PROMPTS
import json
from pprint import pprint
from tqdm import tqdm

import plotly.io as pio
pio.templates.default = "ggplot2"

In [2]:
#models = ["gemma-3-270m-it", "Qwen3-0_6B", "gemma-3-1b-it", "Qwen3-1_7B", "gemma-3-4b-it", "Qwen3-4B", "gemma-3-12b-it", "Qwen3-14B", "gemma-3-27b-it", "Qwen3-32B"]
models = ["Qwen3-8B", "Qwen3-32B"]
#models = ["roberta-base", "roberta-toxicity"]
train_datasets = [    
    ["code-review-dataset-larger"],
    ["code-review-dataset-larger", "neg-code-review-larger"],
    ["gitter_ethereum-labeled-larger"],
    ["gitter_ethereum-labeled-larger", "neg-gitter-ethereum-labeled-larger"],
    ["hatecheck_balanced"],
    ["hatecheck_simple_balanced"],
    ["hatecheck_complex_balanced"],
]

probes = ["LRProbe", "MMProbe"]

val_dataset = "cumstrudel_train_comments_balanced"

groups = ["a", "b2"]
#groups = ["a", "b"]

device = "mps"

subsets = ["toxic", "healthy"]

interventions = ['none', 'add', 'subtract']

batch_size = 32

out = {}
_out = {}

# prepare hidden states to intervene over
config = configparser.ConfigParser()
config.read('config.ini')
total_experiments = len(models) * len(groups) * len(probes) * len(train_datasets) * len(subsets) * len(interventions)

with tqdm(total=total_experiments, desc="Causal Intervention Experiments") as pbar:
    for model_name in models:
        # with open('/Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/results/label_change_intervention_results_new.json', 'r') as f:
        #     data = json.load(f)
        model = load_model(model_name, device)
        #model_group_results = {}
        for group in groups:
            start_layer = eval(config[model_name][f'intervene_layer_{group}'])
            end_layer = eval(config[model_name][f'probe_layer_{group}'])
            noperiod = eval(config[model_name]['noperiod'])

            if noperiod:
                hidden_states = [
                    (layer, -1) for layer in range(start_layer, end_layer + 1)
                ]
            else:
                hidden_states = []
                for layer in range(start_layer, end_layer + 1):
                    hidden_states.append((layer, -1))
                    hidden_states.append((layer, 0))
            probe_intermediates = {}
            for probe_name in probes:
                print('training probe...')
                # get direction along which to intervene
                #if ProbeClass == LRProbe or ProbeClass == MMProbe or ProbeClass == 'random': 
                dataset_intermediates = {}
                for datasets in train_datasets:
                    acts, labels = [], []
                    dataset_name = ""
                    for dataset in datasets:
                        if "hatecheck" in dataset:
                            dataset_name = dataset
                        else:
                            dataset_name += dataset[0]
                        #print(dataset)
                        acts.append(collect_acts(dataset, model_name, end_layer, noperiod=noperiod).to(device))
                        labels.append(t.Tensor(pd.read_csv(f'datasets/{dataset}.csv')['is_toxic'].tolist()).to(device))
                    acts, labels = t.cat(acts), t.cat(labels)
                    ProbeClass = eval(probe_name)
                    if ProbeClass == LRProbe or ProbeClass == MMProbe:
                        probe = ProbeClass.from_data(acts, labels, device=device)
                    elif ProbeClass == 'random':
                        probe = MMProbe.from_data(acts, labels, device=device)
                        probe.direction = t.nn.Parameter(t.randn_like(probe.direction))
                    # elif ProbeClass == CCSProbe:
                    #     acts = collect_acts(args.train_datasets[0], args.model, end_layer, noperiod=noperiod).to(device)
                    #     neg_acts = collect_acts(args.train_datasets[1], args.model, end_layer, noperiod=noperiod).to(device)
                    #     labels = t.Tensor(pd.read_csv(f'datasets/{args.train_datasets[0]}.csv')['label'].tolist()).to(device)
                    #     probe = ProbeClass.from_data(acts, neg_acts, labels=labels, device=device)

                    direction = probe.direction
                    direction = direction.to("mps")
                    true_acts, false_acts = acts[labels==1], acts[labels==0]
                    true_mean, false_mean = true_acts.mean(0), false_acts.mean(0)
                    direction = direction / direction.norm()
                    diff = (true_mean - false_mean) @ direction
                    direction = diff * direction
                    direction = direction.cpu()

                    # set prompt (hardcoded for now)
                    if "roberta" in model_name:
                        prompt = PROMPTS["INTERVENTION_PROMPT_ROBERTA"]
                    else:
                        prompt = PROMPTS["INTERVENTION_PROMPT"]
                    
                    subset_intermediates = {}
                    for subset in subsets:
                        # prepare data
                        queries = prepare_data(prompt, val_dataset, subset=subset)
                        #print(queries)
                        intermediates = {}
                        #print('running intervention experiment...')
                        for intervention in interventions:
                            # do intervention experiment
                            p_diff, tot = intervention_experiment(model, model_name, queries, direction, hidden_states,
                                                                intervention=intervention, batch_size=batch_size)

                            # save results
                            intermediates[intervention] = {
                                'probe class' : probe_name,
                                #'prompt' : prompt,…
                                'p_diff' : p_diff,
                                'tot' : tot,
                                #"direction" : direction,
                                'subset' : subset,
                                'hidden_states' : hidden_states
                            }
                            pbar.update(1)

                        subset_intermediates[subset] = intermediates
                    dataset_intermediates[dataset_name] = subset_intermediates
                probe_intermediates[probe_name] = dataset_intermediates
            #model_group_results[group] = probe_intermediates
        #current_model_data = {model_name: model_group_results}
            _out[group] = probe_intermediates
        out[model_name] = _out
        # data.append(out)
        #data.append(current_model_data)
        #print(data)
        with open(f'/Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/results/label_change_intervention_results_comments_{model_name}_all_probes_only_a_b2.json', 'w') as f:
            json.dump(out, f, indent=4)


# with open('experimental_outputs/label_change_intervention_results.json', 'r') as f:
#     data = json.load(f)
# data.append(out)
# with open('experimental_outputs/label_change_intervention_results.json', 'w') as f:
#     json.dump(data, f, indent=4)

Causal Intervention Experiments:   0%|          | 0/336 [00:00<?, ?it/s]

Loading model Qwen3-8B...
YEAHHH!
training probe...
Directory: /Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/data/acts/Qwen3-8B/code-review-dataset-larger


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Causal Intervention Experiments:   2%|▏         | 6/336 [08:01<7:14:24, 78.98s/it]

Directory: /Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/data/acts/Qwen3-8B/code-review-dataset-larger
Directory: /Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/data/acts/Qwen3-8B/neg-code-review-larger


...

Causal Intervention Experiments:  96%|█████████▋| 324/336 [18:46:34<1:08:26, 342.21s/it]

Directory: /Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/data/acts/Qwen3-32B/hatecheck_simple_balanced


Causal Intervention Experiments:  98%|█████████▊| 330/336 [19:20:56<34:14, 342.44s/it]  

Directory: /Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/data/acts/Qwen3-32B/hatecheck_complex_balanced


Causal Intervention Experiments: 100%|██████████| 336/336 [19:55:19<00:00, 213.45s/it]


In [ ]:
def calculate_nie(pd_toxic_control, pd_healthy_control, pd_toxic_intervention, pd_healthy_intervention, 
                  experiment_type='false_to_true'):
    """
    Calculates Normalized Indirect Effect (NIE) as described in the paper.
    
    NIE measures how effective the intervention was at changing model outputs.
    - NIE = 0: intervention had no effect
    - NIE = 1: intervention completely flipped the model's behavior
    
    Args:
        pd_toxic_control: P(toxic) - P(healthy) for toxic statements without intervention
        pd_healthy_control: P(toxic) - P(healthy) for healthy statements without intervention  
        pd_toxic_intervention: P(toxic) - P(healthy) for toxic statements with intervention
        pd_healthy_intervention: P(toxic) - P(healthy) for healthy statements with intervention
        experiment_type: 'false_to_true' or 'true_to_false'
    
    Returns:
        float: NIE score between 0 and 1
    
    The formulas from the paper are:
    - False→True: (PD^{-}_{*} - PD^{-}) / (PD^{+} - PD^{-})
    - True→False: (PD^{+}_{*} - PD^{+}) / (PD^{-} - PD^{+})
    """
    
    if experiment_type == 'toxic_to_healthy':
        # Measure how much adding the direction to false statements moves them toward true
        intervention_effect = pd_healthy_intervention - pd_healthy_control
        baseline_difference = pd_toxic_control - pd_healthy_control
    elif experiment_type == 'healthy_to_toxic':
        # Measure how much subtracting the direction from true statements moves them toward false
        intervention_effect = pd_toxic_intervention - pd_toxic_control  
        baseline_difference = pd_healthy_control - pd_toxic_control
    else:
        raise ValueError("experiment_type must be 'toxic_to_healthy' or 'healthy_to_toxic'")
    
    # Avoid division by zero
    if abs(baseline_difference) < 1e-8:
        print("Avoided division by zero problem!")
        return 0.0
        
    return float("{:.2f}".format(intervention_effect / baseline_difference))

In [ ]:
with open('/Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/results/label_change_intervention_results.json', 'r') as f:
    data = json.load(f)
model_name = "gemma-3-270m-it"
model_results = data[6][model_name]

nie_results = {}
nie_results[model_name] = {}
for group, group_results in model_results.items():
    nie_results[model_name][group] = {}
    for probe, probe_results in group_results.items():
        print(group + "/" + probe)
        nie_results[model_name][group][probe] = {}
        for dataset, dataset_results in probe_results.items():
            nie_results[model_name][group][probe][dataset] = {}
            measurements = {}
            for subset, subset_results in dataset_results.items():
                print(subset)
                for intervention, intervention_results in subset_results.items():
                    print(intervention)
                    #intervention = intervention_results['intervention']
                    #subset = intervention_results['subset']
                    if intervention == 'none' and subset == 'toxic':
                        measurements['pd_toxic_control'] = intervention_results['p_diff']
                        print("pd_toxic_control: " + str(intervention_results['p_diff']))
                    elif intervention == 'none' and subset == 'healthy':
                        measurements['pd_healthy_control'] = intervention_results['p_diff']
                        print("pd_healthy_control: " + str(intervention_results['p_diff']))
                    elif intervention == 'add' and subset == 'healthy':
                        measurements['pd_healthy_intervention'] = intervention_results['p_diff']
                        print("pd_healthy_intervention: " + str(intervention_results['p_diff']))
                    elif intervention == 'subtract' and subset == 'toxic':
                        measurements['pd_toxic_intervention'] = intervention_results['p_diff']
                        print("pd_toxic_intervention: " + str(intervention_results['p_diff']))
                    
            # Calculate NIEs if we have all required measurements
            if len(measurements) >= 4:
                # False→True NIE (adding direction to false statements)
                if all(k in measurements for k in ['pd_toxic_control', 'pd_healthy_control', 'pd_toxic_intervention']):
                    nie_healthy_to_toxic = calculate_nie(
                        measurements['pd_toxic_control'],
                        measurements['pd_healthy_control'],
                        measurements['pd_toxic_intervention'],
                        None,  # Not needed for this calculation
                        experiment_type="healthy_to_toxic" #'false_to_true'toxic_to_healthy
                    )
                
                # True→False NIE (subtracting direction from true statements)  
                if all(k in measurements for k in ['pd_toxic_control', 'pd_healthy_control', 'pd_healthy_intervention']):
                    nie_toxic_to_healthy = calculate_nie(
                        measurements['pd_toxic_control'],
                        measurements['pd_healthy_control'], 
                        None,  # Not needed for this calculation
                        measurements['pd_healthy_intervention'],
                        experiment_type="toxic_to_healthy"#'true_to_false'
                    )

                        
            nie_results[model_name][group][probe][dataset] = {
                'nie_toxic_to_healthy': nie_toxic_to_healthy,
                'nie_healthy_to_toxic': nie_healthy_to_toxic,
                'raw_measurements': measurements
            }

In [ ]:
for group, group_results in nie_results[model_name].items():
    for probe, probe_results in group_results.items():
        for dataset, dataset_results in probe_results.items():
            print(group + " | " + probe + " | " + dataset + " | " + "nie_toxic_to_healthy: " + str(nie_results[model_name][group][probe][dataset]["nie_toxic_to_healthy"]))
            print(group + " | " + probe + " | " + dataset + " | " + "nie_healthy_to_toxic: " + str(nie_results[model_name][group][probe][dataset]["nie_healthy_to_toxic"]))

In [ ]:
print(nie_results)

In [4]:
#models = ["gemma-3-4b-it", "gemma-3-27b-it", "Qwen3-4B", "Qwen3-14B", "Qwen3-32B"]
models = ["Qwen3-0_6B", "Qwen3-4B", "Qwen3-8B","Qwen3-14B", "Qwen3-32B"]
#models = ["gemma-3-270m-it", "Qwen3-0_6B", "gemma-3-4b-it", "Qwen3-4B"]
#models = ["gemma-3-270m-it", "Qwen3-0_6B", "gemma-3-1b-it", "Qwen3-1_7B", "gemma-3-4b-it", "Qwen3-4B", "gemma-3-12b-it", "Qwen3-14B", "gemma-3-27b-it"]#, "Qwen3-32B"]
#models = ["gemma-3-270m-it", "gemma-3-1b-it", "gemma-3-4b-it", "gemma-3-12b-it", "gemma-3-27b-it"]
#models = ["roberta-base", "roberta-toxicity"]
#models = ["gemma-3-270m-it", "Qwen3-0_6B", "gemma-3-1b-it", "gemma-3-4b-it"]
nie_df = compute_nie_dataframe('/Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/results/label_change_intervention_results_comments_', model_names=models, appendix="_all_probes_only_a_b2")
nie_df.to_csv("experimental_outputs/nie_comments_qwen3_all_probes_a_b2.csv")

Processing model: Qwen3-0_6B
Found Qwen3-0_6B results.
Processing a/LRProbe
Calculating NIE for aLRProbec
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Calculating NIE for aLRProbecn
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Calculating NIE for aLRProbeg
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Calculating NIE for aLRProbegn
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Calculating NIE for aLRProbehatecheck_balanced
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Calculating NIE for aLRProbehatecheck_simple_balanced
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Calculating NIE for aLRProbehatecheck_complex_balanced
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Processing a/MMProbe
Calculating NIE for aMMProbec
Baseline difference: -0.25390625
Baseline difference: 0.25390625
Calculating NIE for aMMProbecn
Baseline difference: -0.25390625
Baseline di

In [ ]:
nie_df_base.style.highlight_max(color = 'red', axis = 0)

In [ ]:
nie_df_tox.style.highlight_(color = 'red', axis = 0)

In [5]:
#nie_df.style.highlight_max(color = 'red', axis = 0)
nie_df = nie_df.style.highlight_min(color = 'blue', axis = 0).highlight_max(color = 'red', axis=0)
nie_df
#nie_df.to_latex("experimental_outputs/causal_qwen.tex", float_format="{:.2f}".format)

In [ ]:
with open('/Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/results/label_change_intervention_results_cleaned.json', 'r') as f:
    data_clean = json.load(f)
with open('/Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/results/label_change_intervention_results.json', 'r') as f:
    data = json.load(f)
out = data_clean
out["roberta-base"] = data[9]["roberta-base"]
out["roberta-toxicity"] = data[9]["roberta-toxicity"]
data_out = out
#print(data_out["Qwen3-32B"])
with open("/Volumes/Samsung SSD 990 PRO 4TB/geometry-of-toxicity/results/label_change_intervention_results_cleaned.json", 'w') as f:
    json.dump(data_out, f, indent=4)
